# Model comparison between PlatoSim and CosmiX

NOTE: To run this notebook you need to install Pyxel v. 1.9.1

### Setup notebook

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

### Imports

In [ ]:
import os
import glob
import h5py
import natsort
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import scipy
import scipy.stats as ss

# Load detector framework
import pyxel
from cosmix.cosmix import run_cosmix
import lmfit 

# PlatoSim
import platosim.referenceFrames as rf
import platosim.plot            as pt
import platosim.utilities       as ut
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_paper
setup_paper()

In [ ]:
# Pixel dimention for squared image
npix = 150

## Distribution of cosmic ray hits

In [ ]:
def phi(x):
    return 1. / np.sqrt(2*np.pi) * np.exp(-x**2 / 2.)
    
def Phi(x): 
#     return scipy.special.erf(x)
    return ss.norm.cdf(x)
    
def getCosmicIntensityFunction(x, psi, omega, alpha, const):
    return 2. / omega * phi((x-psi)/omega) * Phi(alpha*(x-psi)/omega) * const

In [ ]:
# Values read off Fig. 4 from CosmiX paper
xx = np.array([2000, 3000, 4000, 5000, 5500, 6000, 6500, 7000, 7500, 8000, 9000, 10000, 11000, 
               12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000])
yy = [0.002, 0.003, 0.0040, 0.007, 0.011, 0.015, 0.0177, 0.0175, 0.015, 0.013, 0.0100, 0.0075, 0.0055, 
      0.004, 0.003, 0.0025, 0.002, 0.0016, 0.0013, 0.001, 0.0008, 0.0007]

# Select good points for fit
xx0 = xx[3:]
yy0 = yy[3:]

In [ ]:
# Perform LS fit
model = lmfit.Model(getCosmicIntensityFunction)
model.set_param_hint('const', value=100,  min=0.0)
model.set_param_hint('psi',   value=5200, min=0.0)
model.set_param_hint('omega', value=4000, min=0.0)
model.set_param_hint('alpha', value=5,    min=0.0)
fit = model.fit(yy0, x=xx0)

# Store best fit parameters
x = np.linspace(0, 21000, 1000)
y = model.eval(x=x, params=fit.params)

# Fetch standard errors and compute CI
values, stderr = [], []
for param in fit.params.values():
    values.append(param.value)
    stderr.append(param.stderr)
ymax = np.array(values) + 2 * np.array(stderr)
ymin = np.array(values) - 2 * np.array(stderr)
y_max = getCosmicIntensityFunction(x, psi=ymax[0], omega=ymax[1], alpha=ymax[2], const=ymax[3])
y_min = getCosmicIntensityFunction(x, psi=ymin[0], omega=ymin[1], alpha=ymin[2], const=ymin[3])

# Check if model converge and print report
print(f'Model converges: {fit.success}')
print(values)
fit

In [ ]:
# Plot distributions 
fig = plt.figure(figsize=(9.5,5))
plt.plot(xx/1000, yy, 'ko:', label='PLATO CCD data')
plt.plot(xx[:3]/1000, yy[:3], 's', c='deeppink')
plt.plot(x/1000, y, '-', c='royalblue', label='PlatoSim model fit')
plt.fill_between(x/1000, y_min, y_max, color="royalblue", alpha=0.2)
# plt.hist(, bins=10, histtype='step', linewidth=2, facecolor='c', hatch='/', edgecolor='k',fill=True)
# plt.hist(im_cosmic.ravel(), 300, density=True, histtype='step', color='g', linewidth=2, log=False)
# plt.hist(yy, xx, facecolor='b', edgecolor='b', fill=True, alpha=0.3)
plt.xlabel(r'Total charge deposit [ke$^-$]')
plt.ylabel('Normalised counts')
plt.xlim(0, 21)
plt.ylim(0, 0.02)
plt.tight_layout()
plt.legend()
fig.savefig('cosmics_distribution.png', bbox_inches='tight', dpi=200)

### PlatoSim model with Cosmics

In [ ]:
# Initialise PlatoSim
sim = Simulation("output_cosmics_activated", outputDir=os.getcwd())

# Observation
sim["ObservingParameters/NumExposures"] = 1

# Sky
sim["Sky/SkyBackground/UseConstantSkyBackground"] = 'yes'
sim["Sky/SkyBackground/BackgroundValue"]          = -1
sim["Sky/IncludeCosmicsInSubField"]               = True
sim["Sky/Cosmics/CosmicHitRate"]                  = 100
sim["Sky/Cosmics/TrailLength"]                    = [0,300]
sim["Sky/Cosmics/Intensity"]                      = [5232.55981987044, 
                                                     3841.990099449914, 
                                                     5.504757077966231]
# 5232.55981987044, 3841.990099449914, 5.504757077966231, 86.88300697962134

# Platform - make sure no stars are observable
sim["Platform/Orientation/Angles/RAPointing"]  = -100
sim["Platform/Orientation/Angles/DecPointing"] = -100

# Subfield
sim["SubField/NumColumns"] = npix
sim["SubField/NumRows"]    = npix

# Turn off saving and save only relevant parts 
sim.turnOffAllOutput()
sim["ControlHDF5Content/WritePixelMaps"] = True
sim["ControlHDF5Content/WriteCosmics"]   = True

# Turn of quantisation
# sim["CCD/IncludeQuantisation"] = False

# Run simulation
f1 = sim.run(removeOutputFile=True);

# Plot result
fig, ax = f1.showImage(imageNr=0, imgScale="auto", clip=10, figsize=(8,8))

### PlatoSim model without Cosmics

In [ ]:
# Initialise PlatoSim
sim = Simulation("output_cosmics_deactivated", outputDir=os.getcwd())

# Observation
sim["ObservingParameters/NumExposures"] = 1

# Sky
sim["Sky/SkyBackground/UseConstantSkyBackground"] = 'yes'
sim["Sky/SkyBackground/BackgroundValue"]          = -1 
sim["Sky/IncludeCosmicsInSubField"]               = False

# Platform - make sure no stars are observable
sim["Platform/Orientation/Angles/RAPointing"]  = -100
sim["Platform/Orientation/Angles/DecPointing"] = -100

# Subfield
sim["SubField/NumColumns"] = npix
sim["SubField/NumRows"]    = npix

# Turn off saving and save only relevant parts 
sim.turnOffAllOutput()
sim["ControlHDF5Content/WritePixelMaps"] = True
sim["ControlHDF5Content/WriteCosmics"]   = True

# Run simulation
f2 = sim.run(removeOutputFile=True)

# Let's save the image to a FITS image
f2.saveImagesToFITS('output_cosmics_deactivated.fits')

# Show result
fig, ax = f2.showImage(imageNr=0, imgScale="clip", clip=0.5, figsize=(8,8))

### CosmiX model from PlatoSim simulation

In [ ]:
# Load the PLATO input YAML file
config = pyxel.load(os.getcwd() + "/inputfile_pyxel.yaml")

# Setup configurations
exposure = config.exposure
detector = config.ccd_detector
pipeline = config.pipeline

# Set CCD size
detector.geometry.row = npix
detector.geometry.col = npix

# Load a zeros array
pipeline.charge_generation.load_charge.arguments.filename = os.getcwd() + '/output_cosmics_deactivated.fits'
# pipeline.charge_generation.load_charge.arguments.filename = '/lhome/nicholas/software/marvelsim/validation/flat_pyechelle_10s.fits'
pipeline.charge_generation.load_charge.arguments.time_scale = 25
exposure.readout.times = [25]

# Configure dark rate
pipeline.charge_generation.simple_dark_current.arguments.dark_rate = 0.0

# Enable CosmiX
pipeline.photon_generation.cosmix.enabled = True

# Load the model of incident proton hits
pfile = os.getcwd() + '/proton_L2_solarMax_11mm_Shielding.txt'
pipeline.photon_generation.cosmix.arguments.spectrum_file = pfile
# pipeline.photon_generation.cosmix.arguments.spectrum_file = '/lhome/nicholas/software/marvelsim/inputfiles/proton_L2_solarMax_11mm_Shielding.txt
pipeline.photon_generation.cosmix.arguments.initial_energy       = 55
pipeline.photon_generation.cosmix.arguments.particles_per_second = 10

# Run simulations and plot
results = pyxel.exposure_mode(exposure, detector, pipeline)
results

In [ ]:
# Fetch dark image (free of cosmics)
f_bias      = SimFile("output_cosmics_deactivated.hdf5")
im_bias     = f_bias.getImage()
im_bias_med = np.median(im_bias)

# Fetch dark image (free of cosmics)
f_cosmic      = SimFile("output_cosmics_activated.hdf5")
im_cosmic     = f_cosmic.getImage()
im_cosmic_med = np.median(im_cosmic)

# Fetch Pyxel simulation
im_cosmix     = results['image'].to_numpy()[0]
im_cosmix_med = np.median(im_cosmix)

In [ ]:
# run_cosmix(, simulation_mode='cosmic_ray', initial_energy=55, characteristic_length=15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9,6))

# General settings
Nrows, Ncols = im_cosmix.shape
extent = [0, Ncols, 0, Nrows]
cmap   = 'cubehelix'
clabel = 'Normalised counts'                                                                                                                                                                                                                                
origin = 'lower' 

im, norm, vmin, vmax = ut.imageClip(im_cosmic, 'auto', sigma=16)
im0 = ax[0].imshow(im, norm=norm, cmap=cmap, interpolation="nearest", 
                   origin=origin, extent=extent, zorder=0)
divider = make_axes_locatable(ax[0])
cax = divider.new_vertical(size='10%', pad=0.1)
fig.add_axes(cax)
cbar = fig.colorbar(im0, cax=cax, orientation = 'horizontal')
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')

im, norm, vmin, vmax = ut.imageClip(im_cosmix, 'auto', sigma=10)
im1 = ax[1].imshow(im, norm=norm, cmap=cmap, interpolation="nearest", 
                   origin=origin, extent=extent, zorder=0)
divider = make_axes_locatable(ax[1])
cax = divider.new_vertical(size='10%', pad=0.1)
fig.add_axes(cax)
cbar = fig.colorbar(im1, cax=cax, orientation = 'horizontal')
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')

# cbar00 = fig.colorbar(im00, orientation='horizontal', extend='max', shrink=0.84, pad=0.015) 
# cbar00.set_label(clabel, labelpad=3)
# cbar00.ax.tick_params()

fig.text(0.5, 0.94, 'Normalised counts', ha='center')
fig.text(0.5, 0.04, r'Pixel column, $i$', ha='center')
ax[0].set_ylabel(r'Pixel row, $j$')

plt.tight_layout(pad=0)
fig.savefig('cosmics_images.png', bbox_inches='tight', dpi=200)

In [ ]:
# # def plot_loghist(x, bins):
# #     plt.figure()
# #     hist, bins = np.histogram(x, bins=bins)
# #     logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))
# #     plt.hist(x, bins=10)
# #     plt.xscale('log')
# #     plt.yscale('log')

# # plot_loghist(im, 10)
# im = im_cosmic
# bins = np.unique(im)
# fig = plt.figure(figsize=(8,4))
# plt.hist(im_pyxel.ravel(), 1000, density=True, histtype='step', color='b', linewidth=1, log=True);
# plt.hist(im_cosmic.ravel(), 300, density=True, histtype='step', color='g', linewidth=2, log=True);